# 1.0 Verify AWS and Bedrock Access

Run this notebook before building the graph in `1.1_build_graph.ipynb`. It checks the exact AWS paths that Module 1 needs: temporary Vocareum credentials, the `us-east-1` region, Claude Sonnet 4.6, and Amazon Nova multimodal embeddings.

You do not enter AWS keys here. Vocareum injects temporary credentials into the notebook environment. This notebook does not create resources, change permissions, subscribe models, or repair account access.


## Step 1: Confirm the AWS identity and region

The credential check stops instead of letting `boto3` fall back to another local profile. The identity and account number are safe to include in a support request. Secret values are never printed.


In [ ]:
import json
import os

import boto3
from botocore.exceptions import BotoCoreError, ClientError


REQUIRED_REGION = "us-east-1"
region = (
    os.getenv("AWS_REGION")
    or os.getenv("AWS_DEFAULT_REGION")
    or boto3.Session().region_name
    or REQUIRED_REGION
)
session = boto3.Session(region_name=region)
credentials = session.get_credentials()

if credentials is None:
    raise RuntimeError("No AWS credentials are available. Start the Vocareum lab.")

frozen = credentials.get_frozen_credentials()
missing = [
    name
    for name, value in (
        ("access key", frozen.access_key),
        ("secret key", frozen.secret_key),
        ("session token", frozen.token),
    )
    if not value
]
if missing:
    raise RuntimeError("Missing temporary credential fields: " + ", ".join(missing))

identity = session.client("sts", region_name=region).get_caller_identity()
print(f"Account: {identity['Account']}")
print(f"Identity: {identity['Arn']}")
print(f"Region: {region}")

if region != REQUIRED_REGION:
    raise RuntimeError(
        f"This workshop requires {REQUIRED_REGION}; the session selected {region}."
    )

print("PASS: credentials, identity, and region are ready.")


## Step 2: Invoke the models Module 1 uses

This makes three small calls: Sonnet 4.6, streaming Sonnet 4.6, and one 1,024-dimension Nova embedding. A model appearing in the Bedrock catalog is not enough. Only a successful invocation proves the account can run the workshop.


In [ ]:
SONNET_MODEL_ID = "us.anthropic.claude-sonnet-4-6"
NOVA_MODEL_ID = "amazon.nova-2-multimodal-embeddings-v1:0"
EXPECTED_EMBEDDING_DIMENSIONS = 1024
MESSAGES = [
    {"role": "user", "content": [{"text": "Reply with one word: ready"}]}
]
INFERENCE_CONFIG = {"maxTokens": 32}
NOVA_REQUEST = {
    "taskType": "SINGLE_EMBEDDING",
    "singleEmbeddingParams": {
        "embeddingPurpose": "GENERIC_INDEX",
        "embeddingDimension": EXPECTED_EMBEDDING_DIMENSIONS,
        "text": {"truncationMode": "END", "value": "access check"},
    },
}


def failure_kind(error):
    code = error.response.get("Error", {}).get("Code", "Unknown")
    message = error.response.get("Error", {}).get("Message", "")
    lowered = message.lower()
    if "error 002" in lowered or "not allowed for this account" in lowered:
        return "ACCOUNT ENTITLEMENT", code, message
    if code in {"AccessDenied", "AccessDeniedException", "UnauthorizedOperation"}:
        return "IAM OR ORGANIZATION POLICY", code, message
    return "AWS SERVICE ERROR", code, message


def check(label, operation):
    try:
        detail = operation()
    except ClientError as error:
        kind, code, message = failure_kind(error)
        print(f"FAIL: {label}")
        print(f"  {kind}: {code}: {message}")
        return False
    except (BotoCoreError, KeyError, TypeError, ValueError) as error:
        print(f"FAIL: {label}")
        print(f"  CLIENT OR RESPONSE ERROR: {error}")
        return False
    print(f"PASS: {label} - {detail}")
    return True


runtime = session.client("bedrock-runtime", region_name=region)


In [ ]:
def invoke_sonnet():
    response = runtime.converse(
        modelId=SONNET_MODEL_ID,
        messages=MESSAGES,
        inferenceConfig=INFERENCE_CONFIG,
    )
    blocks = response["output"]["message"]["content"]
    text = "".join(block.get("text", "") for block in blocks).strip()
    if not text:
        raise ValueError("Sonnet returned no text")
    return f"answered {text[:40]!r}"


def stream_sonnet():
    response = runtime.converse_stream(
        modelId=SONNET_MODEL_ID,
        messages=MESSAGES,
        inferenceConfig=INFERENCE_CONFIG,
    )
    pieces = []
    for event in response["stream"]:
        delta = event.get("contentBlockDelta", {}).get("delta", {})
        if delta.get("text"):
            pieces.append(delta["text"])
    text = "".join(pieces).strip()
    if not text:
        raise ValueError("streaming Sonnet returned no text")
    return f"answered {text[:40]!r}"


def invoke_nova():
    response = runtime.invoke_model(
        modelId=NOVA_MODEL_ID,
        body=json.dumps(NOVA_REQUEST),
        contentType="application/json",
        accept="application/json",
    )
    payload = json.loads(response["body"].read())
    embeddings = payload.get("embeddings", [])
    vector = embeddings[0].get("embedding", []) if embeddings else []
    if len(vector) != EXPECTED_EMBEDDING_DIMENSIONS:
        raise ValueError(
            f"Nova returned {len(vector)} dimensions; "
            f"expected {EXPECTED_EMBEDDING_DIMENSIONS}"
        )
    return f"returned a {len(vector)}-dimension vector"


results = [
    check("Sonnet 4.6 InvokeModel", invoke_sonnet),
    check("Sonnet 4.6 streaming", stream_sonnet),
    check("Nova multimodal embeddings", invoke_nova),
]

if not all(results):
    raise RuntimeError(
        "Bedrock access is not ready. Copy the account, region, and failing line "
        "into the support request. Do not continue to 1.1."
    )

print("\nEnvironment is ready. Continue to 1.1_build_graph.ipynb.")


## Reading a failure

| Result | Meaning | Owner |
|---|---|---|
| `ACCOUNT ENTITLEMENT` and Error 002 | AWS blocks model use for this allocated account | Vocareum or AWS account-pool administrator |
| `IAM OR ORGANIZATION POLICY` | The session role or an organization policy denied the API action | Workshop template owner or Vocareum |
| Wrong region | The notebook is not using `us-east-1`, where the workshop models run | Lab configuration |
| All checks pass | The account can begin Module 1 | Continue to `1.1_build_graph.ipynb` |

Seeing a model in the Bedrock catalog does not prove it can be invoked. Error 002 is not repaired by changing this notebook, adding AWS keys, or adding another IAM allow statement to the workshop template.
